In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

BASE = Path.cwd()
if not (BASE / "data").exists():
    BASE = BASE.parent
CLEAN = BASE / "data" / "cleaned"
DB_PATH = BASE / "cart2insights.db"

engine = create_engine(f"sqlite:///{DB_PATH}")

date_cols = {
    "orders": ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date"],
    "items": ["shipping_limit_date"],
    "reviews": ["review_creation_date", "review_answer_timestamp"],
}

names = ["customers", "geolocation", "orders", "items", "payments",
         "reviews", "products", "sellers", "category_translation"]

tables = {}
for n in names:
    tables[n] = pd.read_csv(CLEAN / f"{n}.csv",
                            dtype={"customer_zip_code_prefix": str,
                                   "seller_zip_code_prefix": str,
                                   "geolocation_zip_code_prefix": str},
                            parse_dates=date_cols.get(n, []))
    print(n, tables[n].shape)

customers (99441, 5)
geolocation (19015, 5)
orders (99441, 8)
items (112650, 7)
payments (103877, 5)
reviews (98095, 7)
products (32951, 9)
sellers (3095, 4)
category_translation (74, 2)


In [2]:
schema = """
DROP TABLE IF EXISTS payments;
DROP TABLE IF EXISTS reviews;
DROP TABLE IF EXISTS items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS sellers;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS geolocation;
DROP TABLE IF EXISTS category_translation;

CREATE TABLE category_translation (
    product_category_name TEXT PRIMARY KEY,
    product_category_name_english TEXT
);

CREATE TABLE geolocation (
    geolocation_zip_code_prefix TEXT PRIMARY KEY,
    geolocation_lat REAL,
    geolocation_lng REAL,
    geolocation_city TEXT,
    geolocation_state TEXT
);

CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT,
    customer_zip_code_prefix TEXT,
    customer_city TEXT,
    customer_state TEXT
);

CREATE TABLE sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix TEXT,
    seller_city TEXT,
    seller_state TEXT
);

CREATE TABLE products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length REAL,
    product_description_length REAL,
    product_photos_qty REAL,
    product_weight_g REAL,
    product_length_cm REAL,
    product_height_cm REAL,
    product_width_cm REAL,
    FOREIGN KEY (product_category_name) REFERENCES category_translation(product_category_name)
);

CREATE TABLE orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    order_status TEXT,
    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

CREATE TABLE items (
    order_id TEXT NOT NULL,
    order_item_id INTEGER NOT NULL,
    product_id TEXT NOT NULL,
    seller_id TEXT NOT NULL,
    shipping_limit_date TIMESTAMP,
    price REAL,
    freight_value REAL,
    PRIMARY KEY (order_id, order_item_id),
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (seller_id) REFERENCES sellers(seller_id)
);

CREATE TABLE payments (
    order_id TEXT NOT NULL,
    payment_sequential INTEGER NOT NULL,
    payment_type TEXT,
    payment_installments INTEGER,
    payment_value REAL,
    PRIMARY KEY (order_id, payment_sequential),
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);

CREATE TABLE reviews (
    review_id TEXT PRIMARY KEY,
    order_id TEXT NOT NULL,
    review_score INTEGER,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);
"""

with engine.begin() as conn:
    for stmt in schema.split(";"):
        if stmt.strip():
            conn.execute(text(stmt))
print("Tables created")

Tables created


In [3]:
load_order = ["category_translation", "geolocation", "customers", "sellers",
              "products", "orders", "items", "payments", "reviews"]

for n in load_order:
    tables[n].to_sql(n, engine, if_exists="append", index=False, chunksize=5000)
    print("loaded", n, len(tables[n]))

loaded category_translation 74
loaded geolocation 19015
loaded customers 99441
loaded sellers 3095
loaded products 32951
loaded orders 99441
loaded items 112650
loaded payments 103877
loaded reviews 98095


In [4]:
with engine.connect() as conn:
    for n in load_order:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {n}")).scalar()
        print(f"{n:22s} {count:>8}  (csv had {len(tables[n])})")

    conn.execute(text("PRAGMA foreign_keys = ON"))
    problems = conn.execute(text("PRAGMA foreign_key_check")).fetchall()
    print("\nForeign key violations:", len(problems))

category_translation         74  (csv had 74)
geolocation               19015  (csv had 19015)
customers                 99441  (csv had 99441)
sellers                    3095  (csv had 3095)
products                  32951  (csv had 32951)
orders                    99441  (csv had 99441)
items                    112650  (csv had 112650)
payments                 103877  (csv had 103877)
reviews                   98095  (csv had 98095)

Foreign key violations: 0


In [5]:
q = """
SELECT c.customer_state,
       COUNT(DISTINCT o.order_id) AS orders,
       ROUND(SUM(p.payment_value), 2) AS revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY revenue DESC
LIMIT 10;
"""
pd.read_sql(q, engine)

,customer_state,orders,revenue
0,SP,40500,5770266.19
1,RJ,12350,2055690.45
2,MG,11354,1819277.61
3,RS,5345,861802.40
4,PR,4923,781919.55
5,SC,3546,595208.40
6,BA,3256,591270.60
7,DF,2080,346146.17
8,GO,1957,334294.22
9,ES,1995,317682.65


In [6]:
q = """
SELECT
    ROUND(SUM(p.payment_value), 2)            AS total_revenue,
    COUNT(DISTINCT o.order_id)                AS total_orders,
    COUNT(DISTINCT c.customer_unique_id)      AS total_customers,
    (SELECT COUNT(*) FROM sellers)            AS total_sellers,
    ROUND(SUM(p.payment_value) * 1.0 / COUNT(DISTINCT o.order_id), 2) AS avg_order_value,
    (SELECT ROUND(AVG(review_score), 2) FROM reviews) AS avg_review_score
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable');
"""
pd.read_sql(q, engine)

,total_revenue,total_orders,total_customers,total_sellers,avg_order_value,avg_review_score
0,15739137.01,98206,94989,3095,160.27,4.09


In [7]:
q = """
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
       COUNT(DISTINCT o.order_id)                    AS orders,
       ROUND(SUM(p.payment_value), 2)                AS revenue
FROM orders o
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY month
ORDER BY month;
"""
monthly = pd.read_sql(q, engine)
monthly

,month,orders,revenue
0,2016-09,1,136.23
1,2016-10,293,51657.53
2,2016-12,1,19.62
3,2017-01,787,137006.76
4,2017-02,1718,283621.94
5,2017-03,2617,425656.40
6,2017-04,2377,405988.38
7,2017-05,3640,582926.16
8,2017-06,3205,499827.47
9,2017-07,3946,578858.58


In [8]:
q = """
SELECT COALESCE(t.product_category_name_english, 'unknown') AS category,
       COUNT(*)                         AS items_sold,
       ROUND(SUM(i.price), 2)           AS revenue
FROM items i
JOIN products p ON i.product_id = p.product_id
LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
GROUP BY category
HAVING items_sold >= 100
ORDER BY revenue DESC
LIMIT 15;
"""
pd.read_sql(q, engine)

,category,items_sold,revenue
0,health_beauty,9670,1258681.34
1,watches_gifts,5991,1205005.68
2,bed_bath_table,11115,1036988.68
3,sports_leisure,8641,988048.97
4,computers_accessories,7827,911954.32
5,furniture_decor,8334,729762.49
6,cool_stuff,3796,635290.85
7,housewares,6964,632248.66
8,auto,4235,592720.11
9,garden_tools,4347,485256.46


In [9]:
q = """
WITH seller_stats AS (
    SELECT s.seller_id,
           s.seller_state,
           COUNT(DISTINCT i.order_id) AS orders,
           ROUND(SUM(i.price), 2)     AS revenue
    FROM items i
    JOIN sellers s ON i.seller_id = s.seller_id
    GROUP BY s.seller_id, s.seller_state
)
SELECT RANK() OVER (ORDER BY revenue DESC) AS revenue_rank,
       seller_id, seller_state, orders, revenue
FROM seller_stats
ORDER BY revenue_rank
LIMIT 10;
"""
pd.read_sql(q, engine)

,revenue_rank,seller_id,seller_state,orders,revenue
0,1,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1132,229472.63
1,2,53243585a1d6dc2643021fd1853d8905,BA,358,222776.05
2,3,4a3ca9315b744ce9f8e9374361493884,SP,1806,200472.92
3,4,fa1c13f2614d7b5c4749cbc52fecda94,SP,585,194042.03
4,5,7c67e1448b00f6e969d365cea6b010ab,SP,982,187923.89
5,6,7e93a43ef30c4f03f38b393420bc753a,SP,336,176431.87
6,7,da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,160236.57
7,8,7a67c85e85bb2ce8582c35f2203ad736,SP,1160,141745.53
8,9,1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,138968.55
9,10,955fee9216a65b617aa5c0531780ce60,SP,1287,135171.70


In [10]:
q = """
SELECT customer_unique_id, orders, total_spent
FROM (
    SELECT c.customer_unique_id,
           COUNT(DISTINCT o.order_id)     AS orders,
           ROUND(SUM(p.payment_value), 2) AS total_spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY c.customer_unique_id
) AS t
WHERE total_spent > (SELECT AVG(payment_value) * 5 FROM payments)
ORDER BY total_spent DESC
LIMIT 10;
"""
pd.read_sql(q, engine)

,customer_unique_id,orders,total_spent
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31
4,459bef486812aa25204be022145caa62,1,6922.21
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66
6,4007669dec559734d6f53e029e360987,1,6081.54
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34
8,48e1ac109decbb87765a3eade6854098,1,4681.78
9,c8460e4251689ba205045f3ea17884a1,4,4655.91


In [11]:
q = """
WITH m AS (
    SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS month,
           SUM(p.payment_value) AS revenue
    FROM orders o
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status NOT IN ('canceled', 'unavailable')
    GROUP BY month
)
SELECT month,
       ROUND(revenue, 2) AS revenue,
       ROUND(LAG(revenue) OVER (ORDER BY month), 2) AS prev_month,
       ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0
             / LAG(revenue) OVER (ORDER BY month), 1) AS growth_pct
FROM m
ORDER BY month;
"""
pd.read_sql(q, engine)

,month,revenue,prev_month,growth_pct
0,2016-09,136.23,NaN,NaN
1,2016-10,51657.53,136.23,37819.3
2,2016-12,19.62,51657.53,-100.0
3,2017-01,137006.76,19.62,698201.5
4,2017-02,283621.94,137006.76,107.0
5,2017-03,425656.40,283621.94,50.1
6,2017-04,405988.38,425656.40,-4.6
7,2017-05,582926.16,405988.38,43.6
8,2017-06,499827.47,582926.16,-14.3
9,2017-07,578858.58,499827.47,15.8


In [12]:
q = """
SELECT CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 'delayed' ELSE 'on_time' END AS delivery_status,
       COUNT(*)                        AS orders,
       ROUND(AVG(r.review_score), 2)   AS avg_review_score
FROM orders o
JOIN reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY delivery_status;
"""
pd.read_sql(q, engine)

,delivery_status,orders,avg_review_score
0,delayed,7622,2.57
1,on_time,87671,4.30


In [13]:
q = """
WITH cust AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
)
SELECT CASE WHEN orders > 1 THEN 'repeat' ELSE 'one_time' END AS customer_type,
       COUNT(*) AS customers
FROM cust
GROUP BY customer_type;
"""
pd.read_sql(q, engine)

,customer_type,customers
0,one_time,93099
1,repeat,2997
